# Signal Detection Theory Analysis — Experiment 2

Computes **d'** (sensitivity), **criterion c** (response bias), and **AUC** for each model × set-size × feedback condition using:
- `sklearn.metrics.confusion_matrix` — extract TP/FP/TN/FN
- `sklearn.metrics.roc_auc_score`, `roc_curve` — ROC analysis
- `scipy.stats.norm.ppf` — Z-score transformation for d' and c

Signal = *imagined* (internal) | Noise = *perceived* (external)

Conditions: 2 set-sizes (SS=20, SS=40) × 2 feedback levels (No-FB, FB)

**Figures exported:** FigS9 (d', c bars) · FigS10 (ROC curves)

**Referenced in:** Supplemental Materials, Section S4

In [2]:
# ── Imports & project config ───────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import norm

# sklearn — SDT computation & ROC
from sklearn.metrics import (confusion_matrix, roc_auc_score,
                             roc_curve, ConfusionMatrixDisplay)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

import rmllm
PROJ = Path(rmllm.config.PROJ_ROOT)
DATA = PROJ / 'data' / 'processed'
SUP  = PROJ / 'reports' / 'figures' / 'supplemental'
SUP.mkdir(parents=True, exist_ok=True)

DPI = 700

# ── Shared aesthetics (matches manuscript figures) ─────────────────────────
plt.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 11,
    'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.labelsize': 11, 'axes.labelweight': 'bold',
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'axes.linewidth': 1.0, 'axes.facecolor': 'white',
    'figure.facecolor': 'white', 'axes.grid': False,
    'xtick.bottom': True, 'ytick.left': True,
    'xtick.direction': 'out', 'ytick.direction': 'out',
    'xtick.major.size': 4, 'ytick.major.size': 4,
    'legend.fontsize': 9, 'legend.framealpha': 0.9,
    'legend.edgecolor': '#cccccc',
})

MODEL_ORDER  = ['Gemma3:12b','Gemma3:12b-QAT','Gemma3:27b',
                'Gemma3:27b-QAT','Llama3.3:70b','Llama4:16x17b']
MODEL_LABELS = ['G3:12b','G3:12b\nQAT','G3:27b',
                'G3:27b\nQAT','L3.3:70b','L4:16x17b']
MODEL_COLORS = plt.cm.tab10.colors[:6]

def _panel_tag(ax, letter, title=''):
    ax.text(-0.13, 1.07, letter, transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='top', ha='left')
    if title:
        ax.set_title(title, pad=6, fontsize=12, fontweight='bold')

print("Setup complete.")


Setup complete.


In [3]:
# ── SDT helper functions (sklearn + scipy) ────────────────────────────────
def compute_sdt(y_true, y_pred, y_score=None, accuracy=None, confidence=None):
    """
    Compute Type 1 and Type 2 SDT measures for one model/condition.

    Type 1 (source discrimination):
      y_true  : 1 = signal (imagined), 0 = noise (perceived)
      y_pred  : 1 = responded 'internal', 0 = responded 'external'
      y_score : signed confidence score for Type 1 ROC/AUC
                (internal judgment = +confidence, external = -confidence)

    Type 2 (metacognitive sensitivity):
      accuracy   : 1 = correct judgment, 0 = incorrect judgment
      confidence : raw confidence rating (higher = more certain)
      Type 2 AUC (roc_auc_score(accuracy, confidence)) measures how well
      confidence tracks correctness, i.e. metacognitive sensitivity.
      Undefined (NaN) when accuracy is constant (all correct or all wrong).

    Returns dict with keys:
      HR, FA, dprime, c, auc1 (Type 1), auc2 (Type 2), n_signal, n_noise
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # sklearn confusion_matrix: rows=true, cols=pred → [TN FP / FN TP]
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    TN, FP, FN, TP = cm.ravel()

    n_signal = TP + FN   # total imagined trials
    n_noise  = TN + FP   # total perceived trials

    # Log-linear correction (Hautus 1995) — avoids ±inf with HR=1 or FA=0
    HR = (TP + 0.5) / (n_signal + 1)
    FA = (FP + 0.5) / (n_noise  + 1)

    dprime = float(norm.ppf(HR) - norm.ppf(FA))
    c      = float(-0.5 * (norm.ppf(HR) + norm.ppf(FA)))

    # ── Type 1 AUC (source discrimination via signed confidence) ──────────
    auc1 = None
    if y_score is not None and len(np.unique(y_score)) > 1:
        try:
            auc1 = roc_auc_score(y_true, y_score)
        except Exception:
            auc1 = np.nan

    # ── Type 2 AUC (metacognitive: does confidence track correctness?) ─────
    # roc_auc_score(correct/incorrect, confidence)
    # NaN when all trials are correct or all wrong (no variance in accuracy)
    auc2 = None
    if accuracy is not None and confidence is not None:
        acc_arr  = np.asarray(accuracy)
        conf_arr = np.asarray(confidence, dtype=float)
        if len(np.unique(acc_arr)) > 1 and len(np.unique(conf_arr)) > 1:
            try:
                auc2 = roc_auc_score(acc_arr, conf_arr)
            except Exception:
                auc2 = np.nan
        else:
            auc2 = np.nan   # degenerate: all correct or all wrong

    return dict(HR=round(HR,4), FA=round(FA,4),
                dprime=round(dprime,4), c=round(c,4),
                auc1=round(auc1,4) if auc1 is not None else None,
                auc2=round(auc2,4) if (auc2 is not None and not np.isnan(auc2)) else np.nan,
                n_signal=int(n_signal), n_noise=int(n_noise))


def signed_confidence(judgment, confidence):
    """
    Signed discriminant score for Type 1 ROC.
    Internal judgment → +confidence; External → -confidence.
    """
    return np.where(judgment == 'internal',
                    confidence.astype(float),
                   -confidence.astype(float))


print("SDT functions defined (Type 1 + Type 2).")


SDT functions defined (Type 1 + Type 2).


In [4]:
# ── Load & preprocess Exp 2 data ──────────────────────────────────────────
df2 = pd.read_csv(DATA / 'exp2_trial_data.csv')

# Strip 'test:' prefix from source_test
df2['source'] = df2['source_test'].str.replace('test:', '', regex=False)

# Binary labels: imagined = signal (1), perceived = noise (0)
df2['y_true'] = (df2['source'] == 'imagined').astype(int)
df2['y_pred'] = (df2['Judgment_test'] == 'internal').astype(int)

# Signed confidence for ROC
df2['y_score'] = signed_confidence(df2['Judgment_test'], df2['confidence'])

# Condition labels
df2['condition'] = (df2['fb_exp'].map({False:'No-FB', True:'FB'}) + ', SS=' +
                    df2['setsize'].astype(str))

CONDITIONS = ['No-FB, SS=20', 'No-FB, SS=40', 'FB, SS=20', 'FB, SS=40']
COND_PAL   = {'No-FB, SS=20':'#2166ac', 'No-FB, SS=40':'#6baed6',
              'FB, SS=20':   '#d6604d', 'FB, SS=40':   '#f4a582'}

print(f"Exp 2: {len(df2):,} trials | conditions: {sorted(df2['condition'].unique())}")
print(df2.groupby(['condition','source'])['accuracy'].agg(['count','mean']).round(3))


Exp 2: 72,000 trials | conditions: ['FB, SS=20', 'FB, SS=40', 'No-FB, SS=20', 'No-FB, SS=40']
                        count   mean
condition    source                 
FB, SS=20    imagined    6000  0.613
             perceived   6000  0.573
FB, SS=40    imagined   12000  0.521
             perceived  12000  0.557
No-FB, SS=20 imagined    6000  0.383
             perceived   6000  0.779
No-FB, SS=40 imagined   12000  0.386
             perceived  12000  0.748


In [5]:
# ── Compute SDT per model × condition ────────────────────────────────────
rows2 = []
for cond in CONDITIONS:
    for model in MODEL_ORDER:
        sub = df2[(df2['condition'] == cond) & (df2['model'] == model)]
        if sub.empty:
            continue
        res = compute_sdt(sub['y_true'], sub['y_pred'], sub['y_score'],
                          accuracy=sub['accuracy'], confidence=sub['confidence'])
        rows2.append(dict(condition=cond, model=model, **res))

sdt2 = pd.DataFrame(rows2)

pd.set_option('display.float_format', '{:.3f}'.format)
print("\n=== Exp 2 SDT Summary ===")
print(sdt2[['condition','model','n_signal','n_noise',
            'HR','FA','dprime','c','auc1','auc2']].to_string(index=False))
print()
print("Note: auc1 = Type 1 AUC (source); auc2 = Type 2 AUC (metacognitive).")



=== Exp 2 SDT Summary ===
   condition          model  n_signal  n_noise    HR    FA  dprime      c  auc1  auc2
No-FB, SS=20     Gemma3:12b      1000     1000 0.167 0.091   0.367  1.148 0.511 0.496
No-FB, SS=20 Gemma3:12b-QAT      1000     1000 0.551 0.500   0.128 -0.064 0.570 0.581
No-FB, SS=20     Gemma3:27b      1000     1000 0.211 0.042   0.932  1.268 0.586 0.443
No-FB, SS=20 Gemma3:27b-QAT      1000     1000 0.030 0.029   0.030  1.889 0.531 0.533
No-FB, SS=20   Llama3.3:70b      1000     1000 0.894 0.164   2.223 -0.135 0.902 0.649
No-FB, SS=20  Llama4:16x17b      1000     1000 0.447 0.504  -0.143  0.061 0.503 0.566
No-FB, SS=40     Gemma3:12b      2000     2000 0.113 0.056   0.375  1.400 0.521 0.500
No-FB, SS=40 Gemma3:12b-QAT      2000     2000 0.457 0.356   0.260  0.239 0.566 0.536
No-FB, SS=40     Gemma3:27b      2000     2000 0.391 0.258   0.371  0.464 0.556 0.513
No-FB, SS=40 Gemma3:27b-QAT      2000     2000 0.090 0.090   0.003  1.341 0.426 0.432
No-FB, SS=40   Llama3.3:70b

In [6]:
# ── Per-trace SDT for error bars ──────────────────────────────────────────
trace_rows = []
for cond in CONDITIONS:
    for model in MODEL_ORDER:
        sub = df2[(df2['condition'] == cond) & (df2['model'] == model)]
        if sub.empty:
            continue
        for trace_id, tdf in sub.groupby('trace'):
            if len(tdf) < 2:
                continue
            res = compute_sdt(tdf['y_true'], tdf['y_pred'], tdf['y_score'],
                              accuracy=tdf['accuracy'], confidence=tdf['confidence'])
            trace_rows.append(dict(condition=cond, model=model, trace_id=trace_id, **res))

sdt2_traces = pd.DataFrame(trace_rows)

# Compute SEM across traces for each model×condition
sdt2_sem = (sdt2_traces
    .groupby(['condition', 'model'])[['dprime', 'c', 'auc2']]
    .sem()
    .reset_index()
    .rename(columns={'dprime': 'dprime_sem', 'c': 'c_sem', 'auc2': 'auc2_sem'}))

# Per-trace MEANS as bar heights (not pooled)
sdt2_trace_mean = (sdt2_traces
    .groupby(['condition', 'model'])[['dprime', 'c', 'auc2']]
    .mean()
    .reset_index())
sdt2_with_sem = sdt2_trace_mean.merge(sdt2_sem, on=['condition','model'], how='left')
print("Trace-level SEM computed for", len(sdt2_sem), "model×condition combinations")
print(sdt2_with_sem[['condition', 'model', 'dprime', 'dprime_sem', 'c', 'c_sem', 'auc2', 'auc2_sem']].to_string(index=False))

Trace-level SEM computed for 24 model×condition combinations
   condition          model  dprime  dprime_sem      c  c_sem  auc2  auc2_sem
   FB, SS=20     Gemma3:12b  -0.953       0.018 -0.017  0.014 0.525     0.010
   FB, SS=20 Gemma3:12b-QAT  -0.486       0.068 -0.009  0.018 0.463     0.011
   FB, SS=20     Gemma3:27b   0.163       0.042 -0.431  0.026 0.508     0.012
   FB, SS=20 Gemma3:27b-QAT   1.252       0.023  0.004  0.014 0.375     0.010
   FB, SS=20   Llama3.3:70b   2.278       0.035  0.122  0.015 0.336     0.021
   FB, SS=20  Llama4:16x17b   0.606       0.062  0.078  0.010 0.428     0.008
   FB, SS=40     Gemma3:12b  -0.203       0.016  0.028  0.009 0.479     0.006
   FB, SS=40 Gemma3:12b-QAT   0.294       0.019  0.190  0.011 0.500     0.010
   FB, SS=40     Gemma3:27b   0.194       0.023 -0.037  0.034 0.521     0.006
   FB, SS=40 Gemma3:27b-QAT  -0.120       0.029 -0.078  0.027 0.481     0.015
   FB, SS=40   Llama3.3:70b   0.574       0.026  0.302  0.007 0.488     0.006
   

In [7]:
# ── APA-style table ───────────────────────────────────────────────────────
import math
dp_hdr = "d'"
print("\nTable S2. SDT Measures — Experiment 2")
print(f"{'Model':<20} {'Condition':<16} {'HR':>6} {'FA':>6} {dp_hdr:>7} {'c':>7} {'AUC1':>7} {'AUC2':>7}")
print('-' * 86)
for cond in CONDITIONS:
    sub = sdt2[sdt2['condition'] == cond]
    print(f"-- {cond} --")
    for _, row in sub.iterrows():
        a1 = f"{row['auc1']:>7.3f}" if row['auc1'] is not None else f"{'N/A':>7}"
        a2 = f"{row['auc2']:>7.3f}" if (row['auc2'] is not None and not math.isnan(row['auc2'])) else f"{'NaN':>7}"
        print(f"  {row['model']:<18} {row['condition']:<16} "
              f"{row['HR']:>6.3f} {row['FA']:>6.3f} "
              f"{row['dprime']:>7.3f} {row['c']:>7.3f} {a1} {a2}")
print()
print("Note. Negative d' = systematic inversion.")
print("AUC1 = Type 1 (source discrimination). AUC2 = Type 2 (metacognitive sensitivity).")
print("Log-linear correction applied (Hautus, 1995).")



Table S2. SDT Measures — Experiment 2
Model                Condition            HR     FA      d'       c    AUC1    AUC2
--------------------------------------------------------------------------------------
-- No-FB, SS=20 --
  Gemma3:12b         No-FB, SS=20      0.167  0.091   0.367   1.148   0.511   0.496
  Gemma3:12b-QAT     No-FB, SS=20      0.551  0.500   0.128  -0.064   0.570   0.581
  Gemma3:27b         No-FB, SS=20      0.211  0.042   0.932   1.268   0.586   0.443
  Gemma3:27b-QAT     No-FB, SS=20      0.030  0.029   0.030   1.889   0.531   0.533
  Llama3.3:70b       No-FB, SS=20      0.894  0.164   2.223  -0.135   0.902   0.649
  Llama4:16x17b      No-FB, SS=20      0.447  0.504  -0.143   0.061   0.503   0.566
-- No-FB, SS=40 --
  Gemma3:12b         No-FB, SS=40      0.113  0.056   0.375   1.400   0.521   0.500
  Gemma3:12b-QAT     No-FB, SS=40      0.457  0.356   0.260   0.239   0.566   0.536
  Gemma3:27b         No-FB, SS=40      0.391  0.258   0.371   0.464   0.556   0.

In [8]:
# ── Feedback effect on d' and c: delta table ────────────────────────────
delta_dp = "Delta-d'"
delta_c  = "Delta-c"
print("\n=== Feedback effect (FB minus No-FB) on d' and c ===")
print(f"{'Model':<22} {'SS':>4} {delta_dp:>9} {delta_c:>8}")
print('-' * 46)
for ss in [20, 40]:
    nofb = sdt2[sdt2['condition'] == f'No-FB, SS={ss}'].set_index('model')
    fb   = sdt2[sdt2['condition'] == f'FB, SS={ss}'   ].set_index('model')
    for model in MODEL_ORDER:
        if model in nofb.index and model in fb.index:
            dd = fb.loc[model,'dprime'] - nofb.loc[model,'dprime']
            dc = fb.loc[model,'c']      - nofb.loc[model,'c']
            direction = '↑' if dd > 0 else '↓'
            print(f"  {model:<20} {ss:>4} {dd:>+8.3f}{direction}  {dc:>+8.3f}")
    print()
print("Positive Δd' = feedback improved sensitivity.")
print("Note Gemma3:12b negative Δd' = feedback-induced systematic inversion.")



=== Feedback effect (FB minus No-FB) on d' and c ===
Model                    SS  Delta-d'  Delta-c
----------------------------------------------
  Gemma3:12b             20   -1.500↓    -1.170
  Gemma3:12b-QAT         20   -0.704↓    +0.060
  Gemma3:27b             20   -0.670↓    -1.746
  Gemma3:27b-QAT         20   +1.473↑    -1.879
  Llama3.3:70b           20   +0.840↑    +0.409
  Llama4:16x17b          20   +0.830↑    +0.026

  Gemma3:12b             40   -0.597↓    -1.370
  Gemma3:12b-QAT         40   +0.056↑    -0.034
  Gemma3:27b             40   -0.126↓    -0.468
  Gemma3:27b-QAT         40   -0.105↓    -1.411
  Llama3.3:70b           40   -0.719↓    +0.492
  Llama4:16x17b          40   +0.313↑    -0.302

Positive Δd' = feedback improved sensitivity.
Note Gemma3:12b negative Δd' = feedback-induced systematic inversion.


In [9]:
# ── FigS9: SDT measures — 2×3 bar chart — Experiment 2 ──────────────────
# Rows = feedback (No-FB / FB), Cols = metric (d', c, Type2 AUC)
# Within each panel: 2 grouped bars per model (SS=20 vs SS=40)
# Error bars: ±1 SEM across trace IDs (Okabe-Ito colors)

METRICS_BAR = [
    dict(col='dprime', sem_col='dprime_sem', ylabel="Sensitivity ($d'$)",
         null=0.0, ylim=(-1.6, 3.5)),
    dict(col='c',      sem_col='c_sem',      ylabel="Criterion ($c$)",
         null=0.0, ylim=(-0.6, 2.2)),
    dict(col='auc2',   sem_col='auc2_sem',   ylabel="Type 2 AUC",
         null=0.5, ylim=(0.28, 0.72)),
]

FB_ROWS = [
    dict(fb_str='No-FB', row_label='No Feedback'),
    dict(fb_str='FB',    row_label='Feedback'),
]

# SS=20 : Okabe-Ito blue  | SS=40 : Okabe-Ito sky-blue + hatch
SS_STYLES = {
    'SS=20': dict(color='#0072B2', hatch='',    label='Set Size 20'),
    'SS=40': dict(color='#56B4E9', hatch='///', label='Set Size 40'),
}

x       = np.arange(len(MODEL_ORDER))
width   = 0.32
offsets = [-width / 2, width / 2]

MODEL_TICK_LABELS = ['G3:12b', 'G3:12b-\nQAT', 'G3:27b',
                     'G3:27b-\nQAT', 'L3.3:70b', 'L4:16x17b']

fig, axes = plt.subplots(2, 3, figsize=(16, 9), facecolor='white',
                          sharey='col')
fig.subplots_adjust(hspace=0.22, wspace=0.28,
                    bottom=0.14, top=0.92, left=0.07, right=0.98)

for row_i, fb_cfg in enumerate(FB_ROWS):
    fb_str = fb_cfg['fb_str']

    for col_i, mcfg in enumerate(METRICS_BAR):
        ax      = axes[row_i, col_i]
        col     = mcfg['col']
        sem_col = mcfg['sem_col']

        for si, (ss_key, ss_st) in enumerate(SS_STYLES.items()):
            # Match condition: starts with fb_str and contains ss_key
            cond = next(c for c in CONDITIONS
                        if c.startswith(fb_str) and ss_key in c)
            sub  = sdt2_with_sem[sdt2_with_sem['condition'] == cond] \
                       .set_index('model').reindex(MODEL_ORDER)
            vals = sub[col].values.astype(float)
            sems = sub[sem_col].values.astype(float)
            vals = np.where(np.isnan(vals), 0.0, vals)
            sems = np.where(np.isnan(sems), 0.0, sems)

            ax.bar(x + offsets[si], vals, width,
                   color=ss_st['color'], hatch=ss_st['hatch'],
                   edgecolor='white', linewidth=0.6,
                   alpha=0.90, zorder=3, label=ss_st['label'])
            ax.errorbar(x + offsets[si], vals, yerr=sems,
                        fmt='none', color='#222222',
                        capsize=3.5, capthick=1.4, elinewidth=1.4, zorder=4)

        # Null reference
        ax.axhline(mcfg['null'], color='#555555', lw=1.2, ls='--', zorder=2)

        ax.set_xticks(x)
        ax.set_xlim(-0.6, len(MODEL_ORDER) - 0.4)
        ax.spines[['top', 'right']].set_visible(False)
        ax.yaxis.grid(True, lw=0.6, color='#e0e0e0', zorder=0)
        ax.set_axisbelow(True)
        ax.tick_params(axis='y', labelsize=10)

        ax.set_xticklabels(MODEL_TICK_LABELS, fontsize=10,
                           fontweight='bold', rotation=40,
                           ha='right', rotation_mode='anchor')

        # y-label on every column (each column is a different metric,
        # so every panel needs its own label — previously only column 0
        # had one, leaving columns 1-2 with no y-axis identity, especially
        # on the bottom row where the column title isn't repeated either)
        ax.set_ylabel(mcfg['ylabel'], fontsize=12, fontweight='bold')

        # Column title on top row only
        if row_i == 0:
            ax.set_title(mcfg['ylabel'], fontsize=13,
                         fontweight='bold', pad=10)

        # Row label on right side
        if col_i == 2:
            ax.annotate(fb_cfg['row_label'],
                        xy=(1.08, 0.5), xycoords='axes fraction',
                        fontsize=15, fontweight='bold', color='white',
                        rotation=-90, va='center', ha='center',
                        bbox=dict(boxstyle='round,pad=0.5',
                                  facecolor='#333333', edgecolor='none'))

        ax.set_ylim(*mcfg['ylim'])

# Shared legend below figure
handles = [
    plt.Rectangle((0, 0), 1, 1,
                  color=st['color'], hatch=st['hatch'],
                  edgecolor='white', alpha=0.90, label=st['label'])
    for st in SS_STYLES.values()
]
fig.legend(handles=handles, loc='lower center', ncol=2,
           fontsize=12, bbox_to_anchor=(0.5, 0.01),
           framealpha=0.95, edgecolor='#bbbbbb')

for fmt in ('pdf', 'png'):
    fig.savefig(SUP / f'Fig6_exp2_sdt.{fmt}', dpi=DPI,
                bbox_inches='tight', facecolor='white')
print("Saved Fig6_exp2_sdt.pdf / .png")
plt.close('all')


Saved FigS9_exp2_sdt.pdf / .png


In [10]:
# ── FigS10: ROC Curves (per-trace mean ± SEM) — Experiment 2 ──────────────────
# For each condition × model, compute ROC curves on each of the 200
# independent traces, interpolate TPR at a common FPR grid, then plot
# mean curve with ±1 SEM band across traces.

FPR_GRID = np.linspace(0, 1, 101)

def _per_trace_roc(condition, model):
    sub_all = df2[(df2['condition'] == condition) & (df2['model'] == model)]
    tpr_list, auc_list = [], []
    for _, tdf in sub_all.groupby('trace'):
        yt = tdf['y_true'].values
        ys = tdf['y_score'].values
        if len(np.unique(yt)) < 2 or len(np.unique(ys)) < 2:
            continue
        try:
            fpr_t, tpr_t, _ = roc_curve(yt, ys)
            tpr_list.append(np.interp(FPR_GRID, fpr_t, tpr_t))
            auc_list.append(roc_auc_score(yt, ys))
        except Exception:
            continue
    if not tpr_list:
        return None
    arr = np.array(tpr_list)
    n   = len(arr)
    return dict(
        mean_tpr = arr.mean(axis=0),
        sem_tpr  = arr.std(axis=0, ddof=1) / np.sqrt(n),
        mean_auc = float(np.mean(auc_list)),
        sem_auc  = float(np.std(auc_list, ddof=1) / np.sqrt(n)),
        n        = n,
    )

fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharey=True, sharex=True)
fig.subplots_adjust(hspace=0.30, wspace=0.15,
                    bottom=0.09, top=0.94, left=0.09, right=0.97)

for ci, cond in enumerate(CONDITIONS):
    ax = axes[ci // 2, ci % 2]
    _panel_tag(ax, 'abcd'[ci], title=cond)

    for mi, model in enumerate(MODEL_ORDER):
        res = _per_trace_roc(cond, model)
        if res is None:
            continue
        lbl = (f"{MODEL_LABELS[mi].replace(chr(10), ' ')} "
               f"(AUC {res['mean_auc']:.2f}±{res['sem_auc']:.2f})")
        ax.plot(FPR_GRID, res['mean_tpr'],
                color=MODEL_COLORS[mi], linewidth=1.8, label=lbl)
        ax.fill_between(FPR_GRID,
                        res['mean_tpr'] - res['sem_tpr'],
                        res['mean_tpr'] + res['sem_tpr'],
                        color=MODEL_COLORS[mi], alpha=0.15)

    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.9, alpha=0.5, label='Chance')
    ax.set_xlabel('False Alarm Rate', fontsize=11)
    ax.set_ylabel('Hit Rate', fontsize=11)
    ax.legend(fontsize=7.5, loc='lower right', framealpha=0.9)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.spines[['top', 'right']].set_visible(False)

fig.text(0.5, 0.01,
         'Mean ROC ± 1 SEM across N = 200 independent traces per model × condition.',
         ha='center', fontsize=9, color='#555555')

for fmt in ('pdf', 'png'):
    fig.savefig(SUP / f'FigS10_exp2_roc.{fmt}', dpi=DPI,
                bbox_inches='tight', facecolor='white')
print('Saved FigS10_exp2_roc.pdf / .png')
plt.close('all')


Saved FigS10_exp2_roc.pdf / .png
